# NYC TLC Trip Data - 20 Preguntas de Negocio (PSet 02)

Este notebook responde las 20 preguntas del enunciado usando exclusivamente `gold.*`.

In [ ]:
import os
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host=os.getenv('POSTGRES_HOST', 'postgres'),
    port=int(os.getenv('POSTGRES_PORT', '5432')),
    dbname=os.getenv('POSTGRES_DB', 'nyc_trips'),
    user=os.getenv('POSTGRES_USER', 'postgres'),
    password=os.getenv('POSTGRES_PASSWORD', '')
)

def run(sql):
    return pd.read_sql(sql, conn)

## 1) Viajes por mes (2024)
Tablas usadas: `gold.fct_trips`.
Interpretacion: identificar meses con mayor y menor demanda en 2024.

In [ ]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, COUNT(*) AS trips
FROM gold.fct_trips
WHERE pickup_date_key >= '2024-01-01' AND pickup_date_key < '2025-01-01'
GROUP BY 1
ORDER BY 1;
""")

## 2) Viajes por `service_type` y mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: comparar la estacionalidad de yellow vs green.

In [ ]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, service_type, COUNT(*) AS trips
FROM gold.fct_trips
GROUP BY 1, 2
ORDER BY 1, 2;
""")

## 3) Top 10 zonas de pickup (total 2024)
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: identificar zonas de origen con mayor volumen.

In [ ]:
run("""
SELECT z.zone_name, z.borough, COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date_key >= '2024-01-01' AND f.pickup_date_key < '2025-01-01'
GROUP BY 1, 2
ORDER BY trips DESC
LIMIT 10;
""")

## 4) Top 10 zonas de dropoff (total 2024)
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: detectar destinos con mayor concentracion de demanda.

In [ ]:
run("""
SELECT z.zone_name, z.borough, COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.do_zone_key = z.zone_key
WHERE f.pickup_date_key >= '2024-01-01' AND f.pickup_date_key < '2025-01-01'
GROUP BY 1, 2
ORDER BY trips DESC
LIMIT 10;
""")

## 5) Top 5 boroughs por mes (pickup)
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: ver concentracion mensual por borough de origen.

In [ ]:
run("""
WITH base AS (
  SELECT DATE_TRUNC('month', f.pickup_date_key) AS month, z.borough, COUNT(*) AS trips
  FROM gold.fct_trips f
  JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
  GROUP BY 1, 2
), r AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY month ORDER BY trips DESC) AS rn
  FROM base
)
SELECT month, borough, trips
FROM r
WHERE rn <= 5
ORDER BY month, trips DESC;
""")

## 6) Horas pico (top 5 horas) para cada dia de semana
Tablas usadas: `gold.fct_trips`.
Interpretacion: comparar patrones horarios por dia.

In [ ]:
run("""
WITH base AS (
  SELECT pickup_day_of_week, pickup_hour, COUNT(*) AS trips
  FROM gold.fct_trips
  GROUP BY 1, 2
), r AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY pickup_day_of_week ORDER BY trips DESC) AS rn
  FROM base
)
SELECT pickup_day_of_week, pickup_hour, trips
FROM r
WHERE rn <= 5
ORDER BY pickup_day_of_week, trips DESC;
""")

## 7) Distribucion de viajes por dia de semana (ranking)
Tablas usadas: `gold.fct_trips`.
Interpretacion: identificar dias de mayor y menor actividad.

In [ ]:
run("""
SELECT pickup_day_of_week, COUNT(*) AS trips
FROM gold.fct_trips
GROUP BY 1
ORDER BY trips DESC;
""")

## 8) Ingreso total (`total_amount`) por mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: ver tendencia mensual de ingresos.

In [ ]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, SUM(total_amount) AS total_revenue
FROM gold.fct_trips
GROUP BY 1
ORDER BY 1;
""")

## 9) Ingreso total por `service_type` y mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: comparar aporte de ingresos entre servicios.

In [ ]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, service_type, SUM(total_amount) AS total_revenue
FROM gold.fct_trips
GROUP BY 1, 2
ORDER BY 1, 2;
""")

## 10) Tip % promedio por mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: seguir variacion del porcentaje de propina.

In [ ]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month,
       AVG(tip_amount / NULLIF(fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips
WHERE fare_amount > 0
GROUP BY 1
ORDER BY 1;
""")

## 11) Tip % por borough y mes
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: comparar comportamiento de propinas por geografia.

In [ ]:
run("""
SELECT DATE_TRUNC('month', f.pickup_date_key) AS month, z.borough,
       AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.fare_amount > 0
GROUP BY 1, 2
ORDER BY 1, 2;
""")

## 12) Top 10 zonas por ingreso total (pickup)
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: detectar zonas de origen con mayor facturacion.

In [ ]:
run("""
SELECT z.zone_name, z.borough, SUM(f.total_amount) AS total_revenue
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
GROUP BY 1, 2
ORDER BY total_revenue DESC
LIMIT 10;
""")

## 13) Top 10 zonas por tip % (pickup) con minimo N viajes
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: aqui se define N=1000 para evitar ruido estadistico.

In [ ]:
run("""
SELECT z.zone_name, z.borough, COUNT(*) AS trips,
       AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS tip_pct
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.fare_amount > 0
GROUP BY 1, 2
HAVING COUNT(*) >= 1000
ORDER BY tip_pct DESC
LIMIT 10;
""")

## 14) Comparacion cash vs card: viajes, ingreso total, tip %
Tablas usadas: `gold.fct_trips`, `gold.dim_payment_type`.
Interpretacion: comparar comportamiento economico por metodo de pago.

In [ ]:
run("""
SELECT p.payment_type, COUNT(*) AS trips, SUM(f.total_amount) AS total_revenue,
       AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips f
JOIN gold.dim_payment_type p ON f.payment_type_id = p.payment_type_id
WHERE p.payment_type IN ('cash', 'card')
GROUP BY 1
ORDER BY 1;
""")

## 15) Duracion promedio (min) por mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: evaluar si la duracion media cambia por estacionalidad.

In [ ]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, AVG(trip_duration_minutes) AS avg_duration_min
FROM gold.fct_trips
GROUP BY 1
ORDER BY 1;
""")

## 16) Distancia promedio por mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: detectar meses con viajes mas largos/cortos.

In [ ]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, AVG(trip_distance) AS avg_distance
FROM gold.fct_trips
GROUP BY 1
ORDER BY 1;
""")

## 17) Velocidad promedio (mph) por borough y franja horaria
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: comparar eficiencia por zona y momento del dia.

In [ ]:
run("""
SELECT z.borough,
       CASE WHEN f.pickup_hour BETWEEN 6 AND 11 THEN 'morning'
            WHEN f.pickup_hour BETWEEN 12 AND 17 THEN 'afternoon'
            WHEN f.pickup_hour BETWEEN 18 AND 23 THEN 'evening'
            ELSE 'night' END AS time_band,
       AVG(f.trip_distance / NULLIF(f.trip_duration_minutes / 60.0, 0)) AS avg_speed_mph
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.trip_duration_minutes > 0 AND f.trip_distance > 0
GROUP BY 1, 2
ORDER BY 1, 2;
""")

## 18) Percentiles p50 y p90 de duracion por borough
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: medir dispersion y colas de tiempos de viaje.

In [ ]:
run("""
SELECT z.borough,
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY f.trip_duration_minutes) AS p50_duration,
       PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY f.trip_duration_minutes) AS p90_duration
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
GROUP BY 1
ORDER BY 1;
""")

## 19) Top 10 zonas (pickup) por p90 de duracion
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: zonas con mayor cola de viajes largos.

In [ ]:
run("""
SELECT z.zone_name, z.borough,
       PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY f.trip_duration_minutes) AS p90_duration
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
GROUP BY 1, 2
ORDER BY p90_duration DESC
LIMIT 10;
""")

## 20) Top 10 rutas borough->borough por numero de viajes
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: detectar los principales flujos entre boroughs.

In [ ]:
run("""
SELECT pu.borough AS pickup_borough, do.borough AS dropoff_borough, COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_zone pu ON f.pu_zone_key = pu.zone_key
JOIN gold.dim_zone do ON f.do_zone_key = do.zone_key
GROUP BY 1, 2
ORDER BY trips DESC
LIMIT 10;
""")

## Evidencia de Partition Pruning
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: demostrar que el planner no escanea todas las particiones.

In [ ]:
run("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM gold.fct_trips
WHERE pickup_date_key BETWEEN '2024-03-01' AND '2024-03-31';
""")

In [ ]:
run("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM gold.dim_zone
WHERE zone_key = 132;
""")

In [ ]:
conn.close()